In [2]:
import pandas as pd
import numpy as np

# ==============================================================================
# 1. CHARGEMENT DES DONNÉES BRUTES
# ==============================================================================
# On force la colonne 'Phone Nur' en type 'str' pour éviter que les zéros initiaux
# soient supprimés ou convertis en notation scientifique par erreur.
df_brut = pd.read_csv('EvaluationForm.csv', dtype={'Phone Nur': str, 'Email': str})

print("==================================================")
print("📊 RAPPORT DE CHARGEMENT ET NETTOYAGE DES DONNÉES")
print("==================================================")
print(f"• Nombre total de lignes initiales : {df_brut.shape[0]}")

# ==============================================================================
# 2. SUPPRESSION DES DOUBLONS (DEDUPLICATION)
# ==============================================================================
# On identifie et supprime les doublons basés sur l'adresse Email.
# On conserve la dernière entrée ('keep=last') car elle contient les infos les plus récentes.
df_propre = df_brut.drop_duplicates(subset=['Email'], keep='last').copy()
lignes_supprimees = df_brut.shape[0] - df_propre.shape[0]
print(f"• Nombre de lignes dupliquées supprimées : {lignes_supprimees}")
print(f"• Nombre de lignes uniques restantes : {df_propre.shape[0]}")

# ==============================================================================
# 3. CORRECTION ET FORMATAGE DES TYPES
# ==============================================================================
# Remplacement des valeurs manquantes (NaN) dans 'CRS Score' par 0, puis conversion en entier
df_propre['CRS Score'] = df_propre['CRS Score'].fillna(0).astype(int)

# Nettoyage des espaces blancs inutiles dans les colonnes textuelles clés
df_propre['Form Status'] = df_propre['Form Status'].str.strip()
df_propre['Current Country of Residence'] = df_propre['Current Country of Residence'].str.strip()

# ==============================================================================
# 4. SEGMENTATION STRATÉGIQUE (FEATURE ENGINEERING)
# ==============================================================================
# Création d'une nouvelle colonne pour catégoriser les candidats selon leur score CRS.
def classifier_score_crs(score):
    if score >= 450:
        return '1 - Excellent (>=450)'
    elif score >= 350:
        return '2 - Intermédiaire (350-449)'
    else:
        return '3 - Faible (<350)'

df_propre['CRS_Segmentation'] = df_propre['CRS Score'].apply(classifier_score_crs)
print("✅ Phase de nettoyage et de structuration terminée avec succès.")

📊 RAPPORT DE CHARGEMENT ET NETTOYAGE DES DONNÉES
• Nombre total de lignes initiales : 3494
• Nombre de lignes dupliquées supprimées : 285
• Nombre de lignes uniques restantes : 3209
✅ Phase de nettoyage et de structuration terminée avec succès.


In [3]:
# ==============================================================================
# 5. ANALYSE STATISTIQUE ET DISTRIBUTION DES PROFILS
# ==============================================================================
print("==================================================")
print("📈 DISTRIBUTION DES CANDIDATS PAR STATUT ET SCORE")
print("==================================================")

print("\n🔹 Répartition par statut de formulaire (Form Status) :")
print(df_propre['Form Status'].value_counts())

print("\n🔹 Répartition par segment de score CRS :")
print(df_propre['CRS_Segmentation'].value_counts())

print("\n🔹 Top 5 des pays d'origine des candidats :")
print(df_propre['Current Country of Residence'].value_counts().head(5))

📈 DISTRIBUTION DES CANDIDATS PAR STATUT ET SCORE

🔹 Répartition par statut de formulaire (Form Status) :
Form Status
Not Eligible                           1764
Eligible                                679
Eligible if get more Language Score     191
Eligible if has work experience          92
Eligible-EE-not email                    91
Other programs                           85
Eligible-Quebec-not email                64
Duplicated                               55
Close - No money                         54
Close                                    35
Need more info                           34
New Form                                 22
Close-Not interested                     21
Spouse would be eligible                 19
CEC                                       3
Name: count, dtype: int64

🔹 Répartition par segment de score CRS :
CRS_Segmentation
3 - Faible (<350)              1721
2 - Intermédiaire (350-449)     968
1 - Excellent (>=450)           520
Name: count, dtype: int64

🔹 T

In [4]:
 # ==============================================================================
# 6. EXPORTATION DES DONNÉES NETTOYÉES
# ==============================================================================
# Sauvegarde du fichier propre pour l'importation dans SQL Server
df_propre.to_csv('EvaluationForm_Clean.csv', index=False, encoding='utf-8-sig')

print("✅ Fichier 'EvaluationForm_Clean.csv' sauvegardé avec succès !")

✅ Fichier 'EvaluationForm_Clean.csv' sauvegardé avec succès !


In [8]:
import sqlalchemy
import urllib

# ==============================================================================
# 7. IMPORTATION DANS SQL SERVER (ETL FINALE) - PILOTE MODERNE
# ==============================================================================

# 1. Configuration avec le nom de serveur correct et le pilote moderne (ODBC 17)
server_name = "MAHNAZ"
database_name = "Immigration_DB"

# 2. Chaîne de connexion utilisant ODBC Driver 17 (plus stable pour Windows Authentication)
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
)
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 3. Exécution du transfert de données
try:
    df_propre.to_sql(name='Fact_Immigration_Evaluation', con=engine, if_exists='replace', index=False)
    print("✅ [SUCCESS] Les données ont été transférées avec succès dans SQL Server !")
except Exception as e:
    print("❌ [ERROR] Échec du transfert :", e)

✅ [SUCCESS] Les données ont été transférées avec succès dans SQL Server !
